# Deploying Strands Agents to AWS Bedrock AgentCore

This notebook demonstrates how to deploy Strands Agents agent to AWS Bedrock AgentCore using direct boto3 API calls. It also enables you to visualize the agent's decision-making process in the GenAI Observability dashboard in Amazon CloudWatch.

If you're not familiar with the Strands Agents SDK, check out the [official documentation](https://strandsagents.com).

To can access the Amazon Bedrock AgentCore Developer Guide, check out the [AWS documentation](https://docs.aws.amazon.com/bedrock-agentcore/).

## Prerequisites

- Python 3.12 or later
- boto3 1.39 or later
- AWS CLI installed and configured with appropriate permissions

In addition, you should have the following readily available:
- Amazon S3 bucket to package code
- AWS IAM role for automation access
- Knowledge bases are fully synched
- External functions for enterprise systems

## Step 0: Install required packages

Ensure you have the latest versions required installed.

In [ ]:
# Install Zip required by agentcore toolkit CLI
!sudo apt install zip

In [3]:
# Upgrade boto3
!pip install boto3 botocore --upgrade
!pip install bedrock-agentcore strands-agents bedrock-agentcore-starter-toolkit

Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws
  Using cached boto3-1.42.30-py3-none-any.whl.metadata (6.8 kB)
  Using cached botocore-1.42.30-py3-none-any.whl.metadata (5.9 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached boto3-1.42.30-py3-none-any.whl (140 kB)
Using cached botocore-1.42.30-py3-none-any.whl (14.6 MB)
Using cached jmespath-1.0.1-py3-none-any.whl (20 kB)
Using cached s3transfer-0.16.0-py3-none-any.whl (86 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [boto3]32m4/5 [boto3]re]
Looking in indexes: https://pypi.org/simple, https://plugin.us-east-1.prod.workshops.aws
  Using cached bedrock_agentcore-1.2.0-py3-none-any.whl.metadata (7.2 kB)
  Using cached strands_agents-1.22.0-py3-none-any.whl.metadata (17 kB)
  Us

## Step 1: Import required libraries

Import all necessary Python libraries for AWS interactions and file handling.

In [ ]:
import os
import re
import sys
import json
import time
import uuid
import string
import random
import yaml
import zipfile
import tempfile
import collections
import boto3
import botocore
from pathlib import Path

## Step 2: Set required parameters

Update the parameters below using your environment specific details. These parameters will be used throughout the notebook for creating and configuring the agent.

In [ ]:
Knowledge_Base_1_Id = '<provide 1st Knowledge Base Id>'
Knowledge_Base_2_Id = '<provide 2nd Knowledge Base Id>'
SolutionAccessRoleArn = '<provide SolutionAccessRoleArn>'
System_Function_1_Name = '<provide 1st System Function Name>'
System_Function_2_Name = '<provide 2nd System Function Name>'
CodeBucketForAutomationARN = '<provide CodeBucketForAutomationARN>'
Agent_Directory_Name = '<provide agent folder name based on use case - such as pet_store_agent>'

## Step 3: Verify agent code requirements

We'll check that necessary files for the agent exist in the expected locations before proceeding with the deployment.

In [6]:
# Verify AgentCore entrypoint exists
agentcore_entrypoint_file = Path(f"{Agent_Directory_Name}/agentcore_entrypoint.py")
if agentcore_entrypoint_file.exists():
    print(f"AgentCore entrypoint found at {agentcore_entrypoint_file}")
else:
    print(f"AgentCore entrypoint not found at {agentcore_entrypoint_file}")

# Verify requirements file exists
requirements_file = Path(f"{Agent_Directory_Name}/requirements.txt")
if requirements_file.exists():
    print(f"Requirements file found at {requirements_file}")
else:
    print(f"Requirements file not found at {requirements_file}")

AgentCore entrypoint found at pet_store_agent/agentcore_entrypoint.py
Requirements file found at pet_store_agent/requirements.txt


## Step 4: Deploy agent with Amazon Bedrock AgentCore starter toolkit

Let's create a .yaml file for our agent

In [ ]:
session = boto3.Session()
account_id = session.client("sts").get_caller_identity()["Account"]
region = boto3.Session().region_name

agent_runtime_name = "StrandsAgentCoreRuntime"

def create_yaml_file():
    '''Create a .bedrock_agentcore.yaml for AgentCore Runtime Toolkit'''

    yaml_content= f'''default_agent: pet_store_agent
agents:
  pet_store_agent:
    name: {agent_runtime_name}
    entrypoint: ./agentcore_entrypoint.py
    deployment_type: direct_code_deploy
    runtime_type: PYTHON_3_12
    platform: linux/arm64
    container_runtime: null
    source_path: ./
    aws:
      execution_role: {SolutionAccessRoleArn}
      execution_role_auto_create: false
      account: '{account_id}'
      region: {region}
      ecr_repository: null
      ecr_auto_create: false
      s3_path: s3://{CodeBucketForAutomationARN.split(':::')[-1]}
      s3_auto_create: false
      network_configuration:
        network_mode: PUBLIC
        network_mode_config: null
      protocol_configuration:
        server_protocol: HTTP
      observability:
        enabled: true
      lifecycle_configuration:
        idle_runtime_session_timeout: null
        max_lifetime: null
    bedrock_agentcore:
      agent_id: null
      agent_arn: null
      agent_session_id: null
    codebuild:
      project_name: null
      execution_role: null
      source_bucket: null
    memory:
      mode: NO_MEMORY
      memory_id: null
      memory_arn: null
      memory_name: null
      event_expiry_days: 30
      first_invoke_memory_check_done: false
      was_created_by_toolkit: false
    identity:
      credential_providers: []
      workload: null
    aws_jwt:
      enabled: false
      audiences: []
      signing_algorithm: ES384
      issuer_url: null
      duration_seconds: 300
    authorizer_configuration: null
    request_header_configuration: null
    oauth_configuration: null
    api_key_env_var_name: null
    api_key_credential_provider_name: null
    is_generated_by_agentcore_create: false
'''
    yaml_file_path = Path(f"{Agent_Directory_Name}/.bedrock_agentcore.yaml")
    with open(yaml_file_path, 'w') as f:
        f.write(yaml_content)
    return yaml_file_path

yaml_file_path = create_yaml_file()
print(f".bedrock_agentcore.yaml created at {yaml_file_path}")

.bedrock_agentcore.yaml created at pet_store_agent/.bedrock_agentcore.yaml


Let's then create a .env file to reflect required environmental variables for our agent

In [10]:
def create_dotenv_file():
    '''Create a .env for AgentCore Runtime Toolkit'''
    dotenv_content= f'''
AWS_DEFAULT_REGION={region}
KNOWLEDGE_BASE_1_ID={Knowledge_Base_1_Id}
KNOWLEDGE_BASE_2_ID={Knowledge_Base_2_Id}
SYSTEM_FUNCTION_1_NAME={System_Function_1_Name}
SYSTEM_FUNCTION_2_NAME={System_Function_2_Name}
OTEL_PYTHON_DISTRO=aws_distro
OTEL_PYTHON_CONFIGURATOR=aws_configurator
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_TRACES_EXPORTER=otlp
OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group=agents/strands-agent-logs,x-aws-log-stream=default,x-aws-metric-namespace=agents
OTEL_RESOURCE_ATTRIBUTES=service.name=strands-agent
AGENT_OBSERVABILITY_ENABLED=true
'''
    dotenv_file_path = Path(f"{Agent_Directory_Name}/.env")
    with open(dotenv_file_path, 'w') as f:
        f.write(dotenv_content)
    return dotenv_file_path

dotenv_file_path = create_dotenv_file()
print(f".env created at {dotenv_file_path}")

.env created at pet_store_agent/.env


## Step 5: Deploy AgentCore Runtime via CLI

Create the AgentCore Runtime using the Bedrock AgentCore Starter Toolkit

In [ ]:
%cd $Agent_Directory_Name
!agentcore deploy --auto-update-on-conflict --force-rebuild-deps $(grep -v '^#' .env | grep -v '^$' | sed 's/^/--env /' | tr '\n' ' ') 
%cd ..

## Step 6: Check AgentCore Runtime Status

Retrieve the Agent id and Agent ARN from the .yaml file which has these values populated from the above CLI.



In [ ]:
with open(Path(f"{Agent_Directory_Name}/.bedrock_agentcore.yaml"), 'r') as f:
    data = yaml.safe_load(f)

agent_runtime_arn = data['agents']['pet_store_agent']['bedrock_agentcore']['agent_arn']
agent_runtime_id = data['agents']['pet_store_agent']['bedrock_agentcore']['agent_id']

print(f"Agent Runtime id: {agent_runtime_id}")
print(f"Agent ARN: {agent_runtime_arn}")

Monitor the status of the AgentCore Runtime until ready for use.

In [ ]:
agentcore_control_client = boto3.client('bedrock-agentcore-control')
def check_runtime_status(agent_runtime_id):
    """Check the status of the AgentCore Runtime"""
    response = agentcore_control_client.get_agent_runtime(
        agentRuntimeId=agent_runtime_id
    )
    return response['status']

# Wait for the runtime to be ready
print("Waiting for AgentCore Runtime to be ready...")
runtime_status = check_runtime_status(agent_runtime_id)
while runtime_status not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    print(f"Runtime status: {runtime_status}")
    time.sleep(10)
    runtime_status = check_runtime_status(agent_runtime_id)
print(f"Runtime status: {runtime_status}")

## Step 8: Test Agent Runtime Deployment

Send a test prompt to the AgentCore Runtime to verify that the agent is live.

In [ ]:
# Create a client for the AgentCore data plane
agentcore_client = boto3.client('bedrock-agentcore')

# Test the AgentCore Runtime with a sample query
try:
    invoke_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=agent_runtime_arn,
        qualifier="DEFAULT",
        traceId=str(uuid.uuid4()),
        contentType="application/json",
        payload=json.dumps({"prompt": "A new user is asking about the price of Doggy Delights"})
    )
    
    # Process the response
    if "text/event-stream" in invoke_response.get("contentType", ""):
        content = []
        for line in invoke_response["response"].iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    line = line[6:]
                    content.append(line)
        response_text = "\n".join(content)
    else:
        events = []
        for event in invoke_response.get("response", []):
            events.append(event)
        
        # Combine all events to fix truncation
        combined_content = ""
        for event in events:
            combined_content += event.decode("utf-8")
        
        response_text = json.loads(combined_content)
    print(f"{agent_runtime_name} response:")
    print(response_text)
except Exception as e:
    print(f"Error invoking AgentCore Runtime: {str(e)}")

💡 Pro Tip: This notebook enables monitoring and tracing capabilities using AWS OpenTelemetry Python Distro. To visualize the agent's decision-making process and gain insights into its performance, go to the GenAI Observability dashboard in Amazon CloudWatch. Click through the various features of the GenAI observability dashboard to get more detailed information on traces.

## Step 9: Cleanup Resources (Optional)

Clean up the AWS resources created in this notebook to avoid incurring unnecessary charges by uncommenting the last line.

In [ ]:
def cleanup_resources():
    '''Clean up AWS resources created in this notebook.'''  
    # Delete the AgentCore Runtime
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=agent_runtime_id
        )
        print(f"Initiated deletion of AgentCore Runtime: {agent_runtime_id}")
    except Exception as e:
        print(f"Error deleting AgentCore Runtime: {agent_runtime_id}")
    
    
# Uncomment the line below to clean up resources
# cleanup_resources()

## Summary

In this notebook, we demonstrated how to:

1. Package the agent code into a Docker container
2. Deploy the agent to AgentCore Runtime using direct boto3 API calls
3. Test the deployed agent using AgentCore Runtime Endpoint